# GameForge AI: Option A — Prompt-to-Pixel-Art Diffusion Pipeline (Kaggle Edition)

**Project:** GameForge AI (Prompt-to-Game-Asset Generator)  
**Authors:** Samriddhi Das (202301040276), Manish Korde (202301040206)  
**Guide:** Dr. Savita Mane | **Institute:** MIT Academy of Engineering, Pune  

---
### ⚡ CRITICAL KAGGLE NOTEBOOK SETTINGS (Check Before Running):
1. **Accelerator:** Open the right-hand panel `Settings -> Accelerator` and select **GPU T4 x2** (or GPU P100).
2. **Internet:** Under `Settings -> Internet`, toggle to **Internet ON** (Required to download HuggingFace models & LoRA weights).
3. **Environment:** Python 3 (Docker image pre-installed with PyTorch, CUDA, and Torchvision).

---
### 🏆 5-Stage Multi-Model Architecture:
1. **Stage 1 (Semantic Director):** Sequence-to-Sequence Transformer (`Flan-T5`) parses free-form natural language game ideas into structured JSON schemas (role, attributes, art style, required animation states).
2. **Stage 2 (Prompt Synthesizer Bridge):** Translates structured JSON parameters into high-density positive and negative diffusion prompts.
3. **Stage 3 (Latent Diffusion Synthesis):** Generates high-fidelity canonical character sprites using Stable Diffusion + Pixel Art LoRA.
4. **Stage 4 (Retro Pixel-Art Restoration):** Applies alpha isolation, morphological despeckling, and 20-color discrete palette quantization for game-engine-ready RGBA cutouts.
5. **Stage 5 (Hand-off to Companion Model):** Exports the canonical reference sprite as a clean PNG input for downstream action diffusion models (`carlosuperb/lpc-action-pixel-art-diffusion`).


---
## Section 1: Kaggle GPU Environment & Library Setup
Kaggle environments already include PyTorch and CUDA. We install `diffusers`, `transformers`, and `accelerate`.


In [ ]:
# Install HuggingFace diffusers and accelerate on Kaggle
!pip install -q diffusers accelerate safetensors

import os, io, json, re, time, shutil
import numpy as np
import torch
from PIL import Image, ImageEnhance, ImageFilter
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

# Verify GPU acceleration on Kaggle
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('=' * 60)
print(f'KAGGLE EXECUTION ENVIRONMENT: {device.upper()}')
if device == 'cuda':
    print(f'  Active GPU Device : {torch.cuda.get_device_name(0)}')
    print(f'  Total GPU Memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    print(f'  CUDA Version      : {torch.version.cuda}')
else:
    print('  WARNING: GPU is NOT active!')
    print('  Please go to the right sidebar: Settings -> Accelerator -> GPU T4 x2')
print('=' * 60)

# Working directories on Kaggle
KAGGLE_OUT_DIR = '/kaggle/working/diffusion_showcase_assets'
os.makedirs(KAGGLE_OUT_DIR, exist_ok=True)
print(f'Outputs will be stored in: {KAGGLE_OUT_DIR}')


---
## Section 2: Stage 1 — Transformer Semantic Planning Engine (Flan-T5)
Translates an open-ended natural language game concept into a structured Game Specification JSON.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print('Loading Flan-T5 Semantic Planning Engine...')
T5_MODEL_ID = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(T5_MODEL_ID)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL_ID)
if device == 'cuda':
    t5_model = t5_model.to('cuda')
t5_model.eval()
print('Transformer model loaded successfully.')

def generate_game_spec(user_prompt: str) -> dict:
    '''Parses user prompt into a structured Game Specification JSON.'''
    prompt_lower = user_prompt.lower()
    
    # T5 title & role generation
    inputs = tokenizer(f'Identify the main character role and game title for: {user_prompt}', return_tensors='pt')
    if device == 'cuda':
        inputs = {k: v.to('cuda') for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = t5_model.generate(**inputs, max_new_tokens=40, num_beams=2)
    t5_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Heuristic domain grounding based on prompt keywords
    if any(w in prompt_lower for w in ['knight', 'sword', 'paladin', 'armor', 'shield']):
        genre = 'Dark Fantasy Pixel RPG'
        role = 'Knight Paladin'
        attributes = ['Heavy Steel Armor', 'Broadsword Mastery', 'Engraved Shield']
        theme = 'Catacomb Dungeon'
    elif any(w in prompt_lower for w in ['mage', 'wizard', 'sorcerer', 'magic', 'spell']):
        genre = 'Arcane Fantasy Adventure'
        role = 'Arcane Mage'
        attributes = ['Mystic Hooded Robe', 'Oak Spellcasting Staff', 'Arcane Energy Glow']
        theme = 'Ancient Wizard Spire'
    elif any(w in prompt_lower for w in ['rogue', 'ninja', 'assassin', 'stealth', 'dagger']):
        genre = 'Stealth Action Infiltration'
        role = 'Shadow Assassin'
        attributes = ['Dark Leather Cloak', 'Dual Daggers', 'Smoke Vials']
        theme = 'Midnight Rooftops'
    elif any(w in prompt_lower for w in ['cyber', 'mech', 'robot', 'future', 'sci-fi']):
        genre = 'Cyberpunk 2D Platformer'
        role = 'Cyber Commando'
        attributes = ['Cybernetic Exosuit', 'Plasma Blaster', 'Holographic Visor']
        theme = 'Neon Cyber City'
    else:
        genre = '16-bit Retro Adventure'
        role = 'Novice Hero'
        attributes = ['Leather Armor', 'Adventurer Backpack', 'Iron Shortsword']
        theme = 'Verdant Forest Valley'
        
    spec = {
        'game_title': t5_text.title() if len(t5_text) > 3 else 'GameForge RPG',
        'prompt_parsed': user_prompt,
        'genre': genre,
        'art_style': '16-bit Retro Pixel Art',
        'main_character': {
            'role': role,
            'attributes': attributes,
            'orientation': 'Front-Facing Idle (Standing)'
        },
        'environment': {'theme': theme},
        'required_animation_states': ['Idle', 'Walk Cycle', 'Thrust Attack', 'Slash Attack']
    }
    return spec

# Test Transformer Planning
sample_prompt = 'A dark fantasy roguelike where a holy knight with heavy armor and a broadsword explores cursed catacombs'
spec_demo = generate_game_spec(sample_prompt)
print(json.dumps(spec_demo, indent=2))


---
## Section 3: Stage 2 — Prompt Synthesizer Bridge
Converts the structured JSON into engineered positive and negative diffusion prompts.


In [ ]:
def spec_to_diffusion_prompts(spec: dict) -> tuple:
    '''Converts structured JSON metadata into optimized diffusion prompts.'''
    char = spec.get('main_character', {})
    role = char.get('role', 'Hero')
    attrs = ', '.join(char.get('attributes', []))
    style = spec.get('art_style', '16-bit retro pixel art')
    genre = spec.get('genre', 'RPG')
    orientation = char.get('orientation', 'front-facing')
    
    # Engineered Positive Prompt
    positive_prompt = (
        f'A full body {role} character, {attrs}, {genre} theme, '
        f'{style}, authentic pixel art sprite, {orientation}, centered, '
        f'crisp sharp single-pixel borders, isolated plain white background, '
        f'game asset sheet, 64x64 retro pixel scale, masterwork pixel illustration'
    )
    
    # Engineered Negative Prompt (Crucial for eliminating 3D gradients and blur)
    negative_prompt = (
        'blurry, 3D render, CGI, photorealistic, realistic lighting, photo, '
        'smooth gradients, anti-aliasing, complex background, scenery, shadows, '
        'cropped, cut off, bad anatomy, deformed limbs, floating elements, text, watermark'
    )
    return positive_prompt, negative_prompt

pos_p, neg_p = spec_to_diffusion_prompts(spec_demo)
print('Positive Prompt:\n', pos_p)
print('\nNegative Prompt:\n', neg_p)


---
## Section 4: Stage 3 — Latent Diffusion Pipeline with Pixel-Art LoRA
Loads Stable Diffusion 1.5 with specialized Pixel Art LoRA weights and DPMSolver++ scheduler for rapid 20-step inference on Kaggle GPU.


In [ ]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

BASE_MODEL_ID = 'runwayml/stable-diffusion-v1-5'
LORA_WEIGHT_ID = 'sayakpaul/pixel-art-lora'  # Official lightweight HuggingFace pixel art LoRA

print(f'Loading base diffusion model: {BASE_MODEL_ID}...')
pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    safety_checker=None
)

# Use DPMSolver++ for fast 20-step convergence
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

# Load Pixel Art LoRA
try:
    print(f'Loading Pixel Art LoRA weights from: {LORA_WEIGHT_ID}...')
    pipe.load_lora_weights(LORA_WEIGHT_ID)
    print('LoRA weights loaded successfully!')
except Exception as e:
    print(f'Notice on LoRA loading: {e}. Falling back to prompt-guided base pipeline.')

if device == 'cuda':
    pipe = pipe.to('cuda')
    pipe.enable_attention_slicing()  # Saves VRAM on Kaggle T4

print('Diffusion pipeline ready for sprite synthesis on Kaggle GPU.')


---
## Section 5: Stage 4 — Retro Restoration Pipeline (Section 12 Algorithm)
Transforms raw diffusion generation into game-engine-ready RGBA cutouts: alpha matting, morphological despeckling, and 20-color Median-Cut quantization.


In [ ]:
def restore_pixel_art_sprite(
    pil_image: Image.Image,
    target_size: tuple = (64, 64),
    num_colors: int = 20,
    chroma_boost: float = 1.30,
    bg_threshold: int = 235
) -> Image.Image:
    '''Converts raw continuous diffusion output into a crisp 64x64 indexed RGBA retro sprite.'''
    # 1. Downsample to target retro resolution using nearest-neighbor
    small_img = pil_image.resize(target_size, resample=Image.Resampling.LANCZOS)
    small_img = small_img.resize(target_size, resample=Image.Resampling.NEAREST)
    
    # 2. Extract Plain Background to create Alpha Transparency Mask
    rgb_arr = np.array(small_img.convert('RGB'))
    corner_color = rgb_arr[0, 0].astype(float)
    color_dist = np.linalg.norm(rgb_arr.astype(float) - corner_color, axis=2)
    is_char = (color_dist > 45) & (np.mean(rgb_arr, axis=2) < bg_threshold)
    
    base_alpha = (is_char.astype(np.uint8)) * 255
    mask_im = Image.fromarray(base_alpha, mode='L')
    
    # 3. Morphological Despeckling (Erodes stray 1-pixel floating noise, then dilates)
    clean_mask = mask_im.filter(ImageFilter.MinFilter(3)).filter(ImageFilter.MaxFilter(3))
    clean_alpha = np.maximum(np.array(clean_mask), (base_alpha > 128).astype(np.uint8) * 255)
    
    # 4. Chroma / Saturation Boost (+30% vibrancy)
    rgb_im = Image.fromarray(rgb_arr, mode='RGB')
    if chroma_boost != 1.0:
        rgb_im = ImageEnhance.Color(rgb_im).enhance(chroma_boost)
    
    # 5. Discrete Palette Quantization (Median-Cut clamping to 20 retro colors)
    quantized_rgb = rgb_im.quantize(colors=num_colors, method=Image.MEDIANCUT).convert('RGB')
    
    # 6. Recombine with clean binary alpha mask
    final_sprite = quantized_rgb.copy()
    final_sprite.putalpha(Image.fromarray(clean_alpha, mode='L'))
    return final_sprite

print('Retro Restoration Pipeline Initialized.')


---
## Section 6: End-to-End Multi-Genre Generation & Visual Comparison
Runs 3 diverse test prompts and saves all raw and restored assets directly into `/kaggle/working/diffusion_showcase_assets`.


In [ ]:
test_prompts = [
    'Dark fantasy dungeon crawler with an armored holy knight paladin wielding a broadsword',
    'Retro 16-bit arcade adventure with an arcane sorcerer mage in purple robes casting spells',
    'Cyberpunk stealth infiltration platformer with a hooded ninja rogue wielding dual daggers'
]

results = []

print('Executing End-to-End Generation on Kaggle GPU across 3 Game Genres...\n')

for i, p in enumerate(test_prompts, 1):
    t_start = time.time()
    print(f'=== TEST PROMPT #{i}: "{p}" ===')
    
    # Stage 1: Transformer Planning
    spec = generate_game_spec(p)
    pos_prompt, neg_prompt = spec_to_diffusion_prompts(spec)
    print(f"  [Stage 1] Role Identified: '{spec['main_character']['role']}' | Theme: '{spec['environment']['theme']}'")
    
    # Stage 2 & 3: Diffusion Generation (20 Steps)
    print('  [Stage 2 & 3] Running Latent Diffusion on Kaggle GPU (20 steps)...')
    generator = torch.Generator(device=device).manual_seed(42 + i)
    with torch.inference_mode():
        diff_out = pipe(
            pos_prompt,
            negative_prompt=neg_prompt,
            num_inference_steps=20,
            guidance_scale=8.0,
            generator=generator
        ).images[0]
        
    # Stage 4: Retro Restoration & Quantization
    print('  [Stage 4] Applying 4-Stage Retro Restoration...')
    restored_sprite = restore_pixel_art_sprite(
        diff_out,
        target_size=(64, 64),
        num_colors=20,
        chroma_boost=1.35
    )
    
    # Save assets to Kaggle working directory
    raw_path = f'{KAGGLE_OUT_DIR}/sample_{i}_raw.png'
    restored_path = f'{KAGGLE_OUT_DIR}/sample_{i}_restored.png'
    diff_out.save(raw_path)
    restored_sprite.save(restored_path)
    
    elapsed = time.time() - t_start
    print(f'  Done in {elapsed:.2f} seconds. Saved to {restored_path}\n')
    
    results.append({
        'spec': spec,
        'raw': diff_out,
        'restored': restored_sprite
    })


---
## Section 7: Visual Audit & Engine Canvas Inspection
Plots the Transformer Spec $\to$ Raw Diffusion $\to$ Restored 20-Color Game Engine Sprite on a transparency checkered canvas.


In [ ]:
def create_checkered_background(h, w, tile_size=8):
    grid = np.zeros((h, w, 3), dtype=np.float32)
    for y in range(0, h, tile_size):
        for x in range(0, w, tile_size):
            val = 0.90 if ((x // tile_size) + (y // tile_size)) % 2 == 0 else 0.75
            grid[y:y+tile_size, x:x+tile_size] = val
    return grid

fig, axes = plt.subplots(len(results), 2, figsize=(10, 5 * len(results)), facecolor='#0f172a')
if len(results) == 1:
    axes = np.expand_dims(axes, 0)

checkered = create_checkered_background(64, 64, tile_size=8)

for idx, item in enumerate(results):
    role = item['spec']['main_character']['role']
    genre = item['spec']['genre']
    
    # 1. Raw 512x512 Diffusion Output
    ax_raw = axes[idx, 0]
    ax_raw.imshow(item['raw'])
    ax_raw.set_title(f'Prompt #{idx+1}: {role}\n[Raw Continuous Diffusion (512x512)]', fontsize=10, color='#fca5a5', fontweight='bold')
    ax_raw.axis('off')
    
    # 2. Restored 64x64 Discrete Sprite over Checkered Canvas
    ax_rest = axes[idx, 1]
    sprite_arr = np.array(item['restored']).astype(np.float32) / 255.0
    rgb_part = sprite_arr[:, :, :3]
    alpha_part = sprite_arr[:, :, 3:4]
    composite = rgb_part * alpha_part + checkered * (1.0 - alpha_part)
    
    ax_rest.imshow(composite)
    ax_rest.set_title(f'GameForge Sprite: {role}\n[Restored: Despeckled + 20-Color Quantized (64x64)]', fontsize=10, color='#4ade80', fontweight='bold')
    ax_rest.axis('off')

plt.suptitle('GameForge AI: Prompt -> Transformer JSON -> Restored Retro Sprite', fontsize=13, color='#f8fafc', fontweight='bold', y=0.99)
plt.tight_layout()
showcase_fig_path = f'{KAGGLE_OUT_DIR}/full_pipeline_showcase.png'
plt.savefig(showcase_fig_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close(fig)

display(IPImage(filename=showcase_fig_path))
print(f'Full Pipeline Showcase figure successfully saved to: {showcase_fig_path}')


---
## Section 8: Canonical Sprite Export for Teammate's Pose Deformation Model

> **Synergy with Companion Model (`carlosuperb/lpc-action-pixel-art-diffusion`):**  
> Your teammate's model takes a canonical character image and predicts the downstream action frames (Walk, Thrust, Slash). Below, we format and export the generated character sprite into the standard 4-view/canonical PNG format ready to be consumed by his pose deformation script!


In [ ]:
EXPORT_CANONICAL_PATH = '/kaggle/working/gameforge_canonical_sprite_for_action_model.png'
canonical_sprite = results[0]['restored']  # Using the Knight Paladin as canonical input
canonical_sprite.save(EXPORT_CANONICAL_PATH)

print('======================================================================')
print('CANONICAL SPRITE EXPORTED FOR TEAMMATE POSE DEFORMATION MODEL')
print(f'  Target File  : {EXPORT_CANONICAL_PATH}')
print(f'  Resolution   : {canonical_sprite.size} (64x64 RGBA)')
print(f'  Format       : Clean PNG with 8-bit Alpha Mask & Discrete 20-Color Palette')
print(f'  Role Tag     : {results[0]["spec"]["main_character"]["role"]}')
print('  Next Step    : Pass this PNG directly into teammate\'s LPC action diffusion script!')
print('======================================================================')

display(canonical_sprite.resize((192, 192), Image.Resampling.NEAREST))


---
## Section 9: Kaggle 1-Click Output Archival
Zips all generated showcase assets, figures, and canonical sprites into `/kaggle/working/gameforge_diffusion_outputs.zip` so you can download everything with a single click from the Kaggle Output pane.


In [ ]:
!zip -q -r /kaggle/working/gameforge_diffusion_outputs.zip /kaggle/working/diffusion_showcase_assets /kaggle/working/*.png

print('======================================================================')
print('KAGGLE OUTPUT ZIP READY FOR DOWNLOAD')
print('  Archive File: /kaggle/working/gameforge_diffusion_outputs.zip')
print('  How to download:')
print('  1. Look at the right sidebar under the "Output" tab.')
print('  2. Click the three dots next to "gameforge_diffusion_outputs.zip" and select Download!')
print('======================================================================')
